# RAG with Markdown + Azure OpenAI + Azure AI Search

A deliberately explicit RAG pipeline:

```text
Markdown → structure-aware chunking → Azure OpenAI embeddings → Azure AI Search
                                                               ↑
Query → Azure OpenAI embedding → vector search ─────────────────┘
```

The notebook keeps chunking, embedding, indexing, retrieval, and generation separate so the architecture remains visible.


## 0. Prerequisites

Install:

```bash
pip install openai azure-search-documents azure-core python-dotenv
```

Set:

- `AZURE_OPENAI_ENDPOINT`
- `AZURE_OPENAI_API_KEY`
- `AZURE_OPENAI_EMBEDDING_DEPLOYMENT`
- `AZURE_OPENAI_API_VERSION`
- `AZURE_SEARCH_ENDPOINT`
- `AZURE_SEARCH_ADMIN_KEY`
- `AZURE_SEARCH_INDEX_NAME`
- `AZURE_OPENAI_CHAT_DEPLOYMENT` (only for the optional final-answer cell)

The notebook discovers the embedding vector dimension from your deployment.


In [ ]:
import os
import re
import hashlib
from pathlib import Path

from dotenv import load_dotenv
from openai import AzureOpenAI
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    HnswParameters,
    VectorSearchProfile,
    VectorSearchAlgorithmMetric,
)
from azure.search.documents.models import VectorizedQuery

load_dotenv()

AOAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AOAI_KEY = os.environ["AZURE_OPENAI_API_KEY"]
AOAI_EMBEDDING_DEPLOYMENT = os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"]
AOAI_API_VERSION = os.environ["AZURE_OPENAI_API_VERSION"]

SEARCH_ENDPOINT = os.environ["AZURE_SEARCH_ENDPOINT"]
SEARCH_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
INDEX_NAME = os.environ["AZURE_SEARCH_INDEX_NAME"]

aoai = AzureOpenAI(
    azure_endpoint=AOAI_ENDPOINT,
    api_key=AOAI_KEY,
    api_version=AOAI_API_VERSION,
)

credential = AzureKeyCredential(SEARCH_KEY)

index_client = SearchIndexClient(
    endpoint=SEARCH_ENDPOINT,
    credential=credential,
)

search_client = SearchClient(
    endpoint=SEARCH_ENDPOINT,
    index_name=INDEX_NAME,
    credential=credential,
)

print("Clients initialized")


## 1. Load the Markdown document

Put `employee_handbook.md` beside the notebook.


In [ ]:
MD_PATH = Path("employee_handbook.md")

markdown_text = MD_PATH.read_text(encoding="utf-8")

print(markdown_text[:3000])


## 2. Structure-aware chunking

We use Markdown headings as structural boundaries and preserve the heading path as metadata.

For example:

```text
# Leave Policy
## Parental Leave
```

becomes metadata such as:

```text
section_path = Leave Policy > Parental Leave
```

A large paragraph is split further so chunks stay within a reasonable size.


In [ ]:
HEADING_RE = re.compile(r"^(#{1,6})\s+(.+?)\s*$")


def structure_aware_chunks(markdown: str, max_words: int = 180):
    lines = markdown.splitlines()

    hierarchy = []
    paragraphs = []
    current_paragraph = []

    def flush_paragraph():
        nonlocal current_paragraph

        if not current_paragraph:
            return

        text = " ".join(x.strip() for x in current_paragraph).strip()

        if text:
            paragraphs.append((hierarchy.copy(), text))

        current_paragraph = []

    for line in lines:
        match = HEADING_RE.match(line)

        if match:
            flush_paragraph()

            level = len(match.group(1))
            title = match.group(2)

            hierarchy = hierarchy[:level - 1]
            hierarchy.append(title)

        elif not line.strip():
            flush_paragraph()

        else:
            current_paragraph.append(line)

    flush_paragraph()

    chunks = []
    chunk_number = 0

    for path, paragraph in paragraphs:
        words = paragraph.split()

        if len(words) <= max_words:
            pieces = [paragraph]
        else:
            pieces = [
                " ".join(words[i:i + max_words])
                for i in range(0, len(words), max_words)
            ]

        for piece in pieces:
            chunk_number += 1

            chunks.append({
                "chunk_number": chunk_number,
                "text": piece,
                "section_path": " > ".join(path) if path else "Root",
                "heading": path[-1] if path else "Root",
            })

    return chunks


chunks = structure_aware_chunks(markdown_text, max_words=180)

for chunk in chunks:
    print(
        f"\n--- Chunk {chunk['chunk_number']} | "
        f"{chunk['section_path']} ---"
    )
    print(chunk["text"][:700])


## 3. Create embeddings with Azure OpenAI

The interface is:

```text
chunk text → embedding deployment → vector
```

The same embedding deployment will later be used for the user query.


In [ ]:
def embed_texts(texts, batch_size=32):
    embeddings = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        response = aoai.embeddings.create(
            model=AOAI_EMBEDDING_DEPLOYMENT,
            input=batch,
        )

        batch_data = sorted(
            response.data,
            key=lambda item: item.index,
        )

        embeddings.extend(
            item.embedding
            for item in batch_data
        )

    return embeddings


sample_vector = embed_texts(["dimension test"])[0]
VECTOR_DIMENSIONS = len(sample_vector)

print("Embedding dimensions:", VECTOR_DIMENSIONS)


In [ ]:
chunk_texts = [chunk["text"] for chunk in chunks]

chunk_embeddings = embed_texts(chunk_texts)

print("Number of chunks:", len(chunk_embeddings))
print("Vector dimensions:", len(chunk_embeddings[0]))
print("First 5 values:", chunk_embeddings[0][:5])


## 4. Create the Azure AI Search vector index

The important relationship is:

```text
contentVector
      ↓
vector-profile
      ↓
hnsw-config
      ↓
HNSW + cosine metric
```

Azure manages the underlying vector index. We only configure it.


In [ ]:
vector_search = VectorSearch(
    algorithms=[
        HnswAlgorithmConfiguration(
            name="hnsw-config",
            parameters=HnswParameters(
                metric=VectorSearchAlgorithmMetric.COSINE
            ),
        )
    ],
    profiles=[
        VectorSearchProfile(
            name="vector-profile",
            algorithm_configuration_name="hnsw-config",
        )
    ],
)

fields = [
    SimpleField(
        name="id",
        type=SearchFieldDataType.String,
        key=True,
    ),

    SearchableField(
        name="content",
        type=SearchFieldDataType.String,
    ),

    SimpleField(
        name="source",
        type=SearchFieldDataType.String,
        filterable=True,
    ),

    SimpleField(
        name="section_path",
        type=SearchFieldDataType.String,
        filterable=True,
    ),

    SimpleField(
        name="heading",
        type=SearchFieldDataType.String,
        filterable=True,
    ),

    SimpleField(
        name="chunk_number",
        type=SearchFieldDataType.Int32,
        filterable=True,
    ),

    SearchField(
        name="contentVector",
        type=SearchFieldDataType.Collection(
            SearchFieldDataType.Single
        ),
        searchable=True,
        vector_search_dimensions=VECTOR_DIMENSIONS,
        vector_search_profile_name="vector-profile",
    ),
]

index = SearchIndex(
    name=INDEX_NAME,
    fields=fields,
    vector_search=vector_search,
)

created_index = index_client.create_or_update_index(index)

print("Index ready:", created_index.name)


## 5. Store chunks + vectors

Each Azure AI Search document contains:

- chunk text
- structural metadata
- embedding vector

The vector goes into `contentVector`.


In [ ]:
source_name = MD_PATH.name

documents = []

for chunk, embedding in zip(chunks, chunk_embeddings):

    raw_id = (
        f"{source_name}|"
        f"{chunk['section_path']}|"
        f"{chunk['text']}"
    )

    chunk_id = hashlib.sha256(
        raw_id.encode("utf-8")
    ).hexdigest()

    documents.append({
        "id": chunk_id,
        "content": chunk["text"],
        "source": source_name,
        "section_path": chunk["section_path"],
        "heading": chunk["heading"],
        "chunk_number": chunk["chunk_number"],
        "contentVector": embedding,
    })

upload_results = search_client.upload_documents(
    documents=documents
)

for result in upload_results:
    if result.succeeded:
        print("Uploaded:", result.key)
    else:
        print("FAILED:", result.key, result.error_message)


## 6. Query-time vector retrieval

```text
User question
      ↓
Azure OpenAI embedding
      ↓
query vector
      ↓
Azure AI Search vector query
      ↓
Top-K chunks
```


In [ ]:
def retrieve(query: str, top_k: int = 5):
    query_vector = embed_texts([query])[0]

    vector_query = VectorizedQuery(
        vector=query_vector,
        k_nearest_neighbors=top_k,
        fields="contentVector",
    )

    results = search_client.search(
        search_text=None,
        vector_queries=[vector_query],
        select=[
            "content",
            "source",
            "section_path",
            "heading",
            "chunk_number",
        ],
    )

    return list(results)


query = "How much parental leave can an eligible employee take?"

results = retrieve(query, top_k=3)

for i, result in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print("Score:", result["@search.score"])
    print("Section:", result["section_path"])
    print("Content:", result["content"])


## 7. Build the context for the LLM

Vector values are used for retrieval. The retrieved **text** is what we give to the generation model.


In [ ]:
def build_context(results):
    blocks = []

    for i, result in enumerate(results, start=1):
        blocks.append(
            f"[Source {i}]\n"
            f"Document: {result['source']}\n"
            f"Section: {result['section_path']}\n"
            f"Content:\n{result['content']}"
        )

    return "\n\n".join(blocks)


context = build_context(results)

print(context)


## 8. Optional: final answer with Azure OpenAI

Retrieval and generation are intentionally separate.


In [ ]:
CHAT_DEPLOYMENT = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")


def answer_with_rag(query: str, results):
    if not CHAT_DEPLOYMENT:
        raise ValueError(
            "Set AZURE_OPENAI_CHAT_DEPLOYMENT first."
        )

    context = build_context(results)

    prompt = f"""
Answer the user's question using only the supplied context.

If the context does not contain enough information,
say that you do not have enough information.

User question:
{query}

Context:
{context}
"""

    response = aoai.chat.completions.create(
        model=CHAT_DEPLOYMENT,
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer questions using the retrieved "
                    "company documentation."
                ),
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
    )

    return response.choices[0].message.content


# Uncomment after setting AZURE_OPENAI_CHAT_DEPLOYMENT:
#
# answer = answer_with_rag(query, results)
# print(answer)


## 9. Delta / incremental ingestion

The chunk IDs are deterministic hashes. This gives us a starting point for incremental ingestion.

Conceptually:

```text
new chunk ID              → upload
existing unchanged ID     → skip
changed chunk             → upload new version + remove old
deleted chunk             → delete
```

A production pipeline would also maintain document versions/hashes and handle cases where a document change causes re-chunking.


In [ ]:
def document_hash(text: str) -> str:
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


print("Document hash:", document_hash(markdown_text))


## 10. Final architecture

```text
                    OFFLINE / INGESTION

employee_handbook.md
        ↓
Structure-aware chunking
        ↓
chunks + metadata
        ↓
Azure OpenAI embedding deployment
        ↓
vectors
        ↓
Azure AI Search
  ├── chunk text
  ├── metadata
  ├── vectors
  └── HNSW vector index


                     ONLINE / QUERY

User question
        ↓
Azure OpenAI embedding deployment
        ↓
query vector
        ↓
Azure AI Search vector search
        ↓
Top-K chunks
        ↓
Context construction
        ↓
Azure OpenAI chat model
        ↓
Answer
```

### Responsibilities

**Your application**
- reads Markdown
- performs structure-aware chunking
- retains metadata
- calls the embedding model
- chooses retrieval parameters
- constructs the prompt

**Azure OpenAI**
- creates embeddings
- optionally generates the final answer

**Azure AI Search**
- stores chunks, metadata, and vectors
- maintains the vector index
- performs vector retrieval
